<a href="https://colab.research.google.com/github/nickitkaproshin-lab/Proshin-liesegang-rings-prediction/blob/main/Proshin_liesegang_rings_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""
Liesegang Rings Prediction Model – финальная версия с интерполяцией вязкости
- Обучение общей модели (все гели)
- Интерполяция вязкости для желатина и агарозы (линейная)
- Учёт стехиометрии через nu_in, nu_out
- Вывод коэффициентов модели
- Два режима: использование обученной модели или ручной ввод коэффициентов
- Интерактивный интерфейс (ipywidgets)
"""

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, mean_absolute_error, r2_score, confusion_matrix

from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# =========================== 1. ЗАГРУЗКА ДАТАСЕТА ===========================
print("Загрузите ваш CSV-файл (liesegang_dataset.csv или .xlsx):")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith('.csv'):
    df = pd.read_csv(filename, encoding='utf-8-sig')
else:
    df = pd.read_excel(filename)

print(f"✅ Загружено {df.shape[0]} строк.")
df = df.dropna(how='all')

# =========================== 2. ВСТРОЕННЫЕ СПРАВОЧНИКИ ===========================
D_water_25 = {
    'Ag+': 1.65e-9, 'Cu2+': 0.72e-9, 'Co2+': 0.73e-9, 'Ni2+': 0.68e-9,
    'Mg2+': 0.71e-9, 'Ca2+': 0.79e-9, 'Zn2+': 0.70e-9, 'Cd2+': 0.72e-9,
    'Pb2+': 0.94e-9, 'Hg2+': 0.85e-9, 'Mn2+': 0.69e-9, 'Fe2+': 0.72e-9,
    'Fe3+': 0.60e-9, 'Al3+': 0.54e-9, 'Cr3+': 0.59e-9, 'La3+': 0.62e-9,
    'NH4+': 1.96e-9, 'Sr2+': 0.79e-9, 'Ba2+': 0.85e-9,
    'OH-': 5.27e-9, 'Cl-': 2.03e-9, 'Br-': 2.08e-9, 'I-': 2.04e-9,
    'SCN-': 1.33e-9, 'CrO4_2-': 1.07e-9, 'Cr2O7_2-': 1.00e-9, 'SO4_2-': 1.06e-9,
    'CO3_2-': 0.92e-9, 'C2O4_2-': 0.83e-9, 'PO4_3-': 0.61e-9, 'HPO4_2-': 0.77e-9,
    'H2PO4-': 0.87e-9, 'Fe(CN)6_4-': 0.76e-9, 'Fe(CN)6_3-': 0.87e-9,
    'SiO3_2-': 0.73e-9, 'MoO4_2-': 0.91e-9, 'WO4_2-': 0.88e-9, 'F-': 1.47e-9,
    'тартрат-': 0.80e-9, 'O2-': 2.0e-9, 'S2-': 1.8e-9
}

ion_radii = {
    'Ag+': 1.15, 'Cu2+': 0.73, 'Co2+': 0.745, 'Ni2+': 0.69, 'Mg2+': 0.72,
    'Ca2+': 1.00, 'Zn2+': 0.74, 'Cd2+': 0.95, 'Pb2+': 1.19, 'Hg2+': 1.02,
    'Mn2+': 0.83, 'Fe2+': 0.78, 'Fe3+': 0.645, 'Al3+': 0.535, 'Cr3+': 0.615,
    'La3+': 1.032, 'NH4+': 1.48, 'Sr2+': 1.18, 'Ba2+': 1.35,
    'OH-': 1.40, 'Cl-': 1.81, 'Br-': 1.96, 'I-': 2.20, 'SCN-': 2.15,
    'CrO4_2-': 2.40, 'Cr2O7_2-': 2.50, 'SO4_2-': 2.30, 'CO3_2-': 1.78,
    'C2O4_2-': 2.12, 'PO4_3-': 2.38, 'HPO4_2-': 2.20, 'H2PO4-': 2.00,
    'Fe(CN)6_4-': 4.00, 'Fe(CN)6_3-': 4.00, 'SiO3_2-': 2.60,
    'MoO4_2-': 2.54, 'WO4_2-': 2.60, 'F-': 1.33, 'тартрат-': 2.50,
    'O2-': 1.40, 'S2-': 1.84
}

ion_charges = {
    'Ag+': 1, 'Cu2+': 2, 'Co2+': 2, 'Ni2+': 2, 'Mg2+': 2, 'Ca2+': 2,
    'Zn2+': 2, 'Cd2+': 2, 'Pb2+': 2, 'Hg2+': 2, 'Mn2+': 2, 'Fe2+': 2,
    'Fe3+': 3, 'Al3+': 3, 'Cr3+': 3, 'La3+': 3, 'NH4+': 1, 'Sr2+': 2, 'Ba2+': 2,
    'OH-': 1, 'Cl-': 1, 'Br-': 1, 'I-': 1, 'SCN-': 1, 'CrO4_2-': 2,
    'Cr2O7_2-': 2, 'SO4_2-': 2, 'CO3_2-': 2, 'C2O4_2-': 2, 'PO4_3-': 3,
    'HPO4_2-': 2, 'H2PO4-': 1, 'Fe(CN)6_4-': 4, 'Fe(CN)6_3-': 3,
    'SiO3_2-': 2, 'MoO4_2-': 2, 'WO4_2-': 2, 'F-': 1, 'тартрат-': 1,
    'O2-': 2, 'S2-': 2
}

# Функция интерполяции вязкости геля (линейная)
def get_viscosity(gel_type, conc):
    """
    gel_type: 'желатин', 'агароза', 'силикагель'
    conc: концентрация в % (0 для силикагеля)
    """
    if gel_type == 'желатин':
        # Значения: при 5% → 1.8, при 10% → 2.5
        if conc <= 5:
            return 1.8
        elif conc <= 10:
            return 1.8 + (2.5 - 1.8) * (conc - 5) / 5
        else:
            # Экстраполяция для >10%
            return 1.8 + (2.5 - 1.8) * (conc - 5) / 5
    elif gel_type == 'агароза':
        # Значения: при 1% → 1.3, при 2% → 1.6
        if conc <= 1:
            return 1.3
        elif conc <= 2:
            return 1.3 + (1.6 - 1.3) * (conc - 1)
        else:
            # Экстраполяция для >2%
            return 1.3 + (1.6 - 1.3) * (conc - 1)
    else:  # силикагель
        return 2.0

# Стехиометрические коэффициенты для обучения
stoichiometry = {
    'Ag2Cr2O7': (2, 1), 'Ag2CrO4': (2, 1), 'CuCrO4': (1, 1), 'CuCr2O7': (1, 2),
    'PbCrO4': (1, 1), 'PbCr2O7': (1, 2), 'BaCrO4': (1, 1), 'SrCrO4': (1, 1),
    'CdCrO4': (1, 1), 'ZnCrO4': (1, 1), 'HgCrO4': (1, 1), 'Cu+Zn/CrO4': (1, 1),
    'Cu+Pb/CrO4': (1, 1), 'Co(OH)2': (1, 2), 'Mg(OH)2': (1, 2), 'Ca(OH)2': (1, 2),
    'Ca(OH)2/CaCO3': (1, 1), 'Ca3(PO4)2': (3, 2), 'Ca5(PO4)3OH': (5, 3),
    'CaHPO4': (1, 1), 'Ca8(HPO4)2(PO4)4': (8, 6), 'Ni(OH)2': (1, 2),
    'Ni+Co(OH)2': (1, 2), 'Cu(OH)2': (1, 2), 'Fe(OH)3': (1, 3), 'Fe(OH)2': (1, 2),
    'Al(OH)3': (1, 3), 'Cr(OH)3': (1, 3), 'Mn(OH)2': (1, 2), 'Zn(OH)2': (1, 2),
    'Cd(OH)2': (1, 2), 'Pb(OH)2': (1, 2), 'Co+Mg(OH)2': (1, 2), 'Co+Ni(OH)2': (1, 2),
    'Co+Cu(OH)2': (1, 2), 'Mg+Ca/PO4': (1, 1), 'F-apatite': (5, 3),
    'PbI2': (1, 2), 'AgCl': (1, 1), 'AgBr': (1, 1), 'AgI': (1, 1),
    'HgI2': (1, 2), 'CuI': (1, 1), 'CdI2': (1, 2), 'PbBr2': (1, 2),
    'HgBr2': (1, 2), 'AgSCN': (1, 1), 'Pb(SCN)2': (1, 2), 'PbI2+HgI2': (1, 2),
    'CuS': (1, 1), 'CdS': (1, 1), 'CaC2O4': (1, 1), 'BaC2O4': (1, 1),
    'CdC2O4': (1, 1), 'CaCO3': (1, 1), 'BaCO3': (1, 1), 'SrCO3': (1, 1),
    'SrSO4': (1, 1), 'CaSO4': (1, 1), 'РЗЭ тартраты': (1, 1),
    'Fe4[Fe(CN)6]3': (4, 3), 'Cu2[Fe(CN)6]': (2, 1), 'NH4Cl': (1, 1),
    'Ag2CrO4+Ag4[Fe(CN)6]': (2, 1), 'PbCrO4+Pb2[Fe(CN)6]': (1, 1),
    'асфальтены': (1, 1), 'CaSiO3': (1, 1), 'CuSiO3': (1, 1),
    'MnO2': (1, 2), 'Ag2MoO4': (2, 1), 'Ag2WO4': (2, 1), 'CaWO4': (1, 1),
    'La2(MoO4)3': (2, 3)
}

# Маппинг системы -> (катион, анион) для обучения
mapping = {
    'Ag2Cr2O7': ('Ag+', 'Cr2O7_2-'), 'Ag2CrO4': ('Ag+', 'CrO4_2-'),
    'CuCrO4': ('Cu2+', 'CrO4_2-'), 'CuCr2O7': ('Cu2+', 'Cr2O7_2-'),
    'PbCrO4': ('Pb2+', 'CrO4_2-'), 'PbCr2O7': ('Pb2+', 'Cr2O7_2-'),
    'BaCrO4': ('Ba2+', 'CrO4_2-'), 'SrCrO4': ('Sr2+', 'CrO4_2-'),
    'CdCrO4': ('Cd2+', 'CrO4_2-'), 'ZnCrO4': ('Zn2+', 'CrO4_2-'),
    'HgCrO4': ('Hg2+', 'CrO4_2-'), 'Cu+Zn/CrO4': ('Cu2+', 'CrO4_2-'),
    'Cu+Pb/CrO4': ('Cu2+', 'CrO4_2-'), 'Co(OH)2': ('Co2+', 'OH-'),
    'Mg(OH)2': ('Mg2+', 'OH-'), 'Ca(OH)2': ('Ca2+', 'OH-'),
    'Ca(OH)2/CaCO3': ('Ca2+', 'CO3_2-'), 'Ca3(PO4)2': ('Ca2+', 'PO4_3-'),
    'Ca5(PO4)3OH': ('Ca2+', 'PO4_3-'), 'CaHPO4': ('Ca2+', 'HPO4_2-'),
    'Ca8(HPO4)2(PO4)4': ('Ca2+', 'PO4_3-'), 'Ni(OH)2': ('Ni2+', 'OH-'),
    'Ni+Co(OH)2': ('Ni2+', 'OH-'), 'Cu(OH)2': ('Cu2+', 'OH-'),
    'Fe(OH)3': ('Fe3+', 'OH-'), 'Fe(OH)2': ('Fe2+', 'OH-'),
    'Al(OH)3': ('Al3+', 'OH-'), 'Cr(OH)3': ('Cr3+', 'OH-'),
    'Mn(OH)2': ('Mn2+', 'OH-'), 'Zn(OH)2': ('Zn2+', 'OH-'),
    'Cd(OH)2': ('Cd2+', 'OH-'), 'Pb(OH)2': ('Pb2+', 'OH-'),
    'Co+Mg(OH)2': ('Co2+', 'OH-'), 'Co+Ni(OH)2': ('Co2+', 'OH-'),
    'Co+Cu(OH)2': ('Co2+', 'OH-'), 'Mg+Ca/PO4': ('Mg2+', 'PO4_3-'),
    'F-apatite': ('Ca2+', 'F-'), 'PbI2': ('Pb2+', 'I-'),
    'AgCl': ('Ag+', 'Cl-'), 'AgBr': ('Ag+', 'Br-'), 'AgI': ('Ag+', 'I-'),
    'HgI2': ('Hg2+', 'I-'), 'CuI': ('Cu+', 'I-'), 'CdI2': ('Cd2+', 'I-'),
    'PbBr2': ('Pb2+', 'Br-'), 'HgBr2': ('Hg2+', 'Br-'), 'AgSCN': ('Ag+', 'SCN-'),
    'Pb(SCN)2': ('Pb2+', 'SCN-'), 'PbI2+HgI2': ('Pb2+', 'I-'),
    'CuS': ('Cu2+', 'S2-'), 'CdS': ('Cd2+', 'S2-'), 'CaC2O4': ('Ca2+', 'C2O4_2-'),
    'BaC2O4': ('Ba2+', 'C2O4_2-'), 'CdC2O4': ('Cd2+', 'C2O4_2-'),
    'CaCO3': ('Ca2+', 'CO3_2-'), 'BaCO3': ('Ba2+', 'CO3_2-'),
    'SrCO3': ('Sr2+', 'CO3_2-'), 'SrSO4': ('Sr2+', 'SO4_2-'),
    'CaSO4': ('Ca2+', 'SO4_2-'), 'РЗЭ тартраты': ('La3+', 'тартрат-'),
    'Fe4[Fe(CN)6]3': ('Fe3+', 'Fe(CN)6_4-'), 'Cu2[Fe(CN)6]': ('Cu2+', 'Fe(CN)6_4-'),
    'NH4Cl': ('NH4+', 'Cl-'), 'Ag2CrO4+Ag4[Fe(CN)6]': ('Ag+', 'CrO4_2-'),
    'PbCrO4+Pb2[Fe(CN)6]': ('Pb2+', 'CrO4_2-'), 'асфальтены': ('C', 'C'),
    'CaSiO3': ('Ca2+', 'SiO3_2-'), 'CuSiO3': ('Cu2+', 'SiO3_2-'),
    'MnO2': ('Mn2+', 'O2-'), 'Ag2MoO4': ('Ag+', 'MoO4_2-'), 'Ag2WO4': ('Ag+', 'WO4_2-'),
    'CaWO4': ('Ca2+', 'WO4_2-'), 'La2(MoO4)3': ('La3+', 'MoO4_2-')
}
for sys in df['Система'].unique():
    if sys not in mapping:
        print(f"⚠️ Система '{sys}' не найдена в mapping, будет пропущена.")
        mapping[sys] = (np.nan, np.nan)

# =========================== 3. ФУНКЦИИ РАСЧЁТА ПРИЗНАКОВ ===========================
def parse_gel(gel_str):
    if pd.isna(gel_str):
        return 'unknown', 0.0
    s = str(gel_str).lower().strip()
    if 'желатин' in s:
        m = re.search(r'(\d+(?:\.\d+)?)%', s)
        conc = float(m.group(1)) if m else 5.0
        return 'желатин', conc
    elif 'агароза' in s:
        m = re.search(r'(\d+(?:\.\d+)?)%', s)
        conc = float(m.group(1)) if m else 1.0
        return 'агароза', conc
    elif 'силикагель' in s:
        return 'силикагель', 0.0
    else:
        return s, 0.0

def compute_D_gel(ion, T_C, gel_type, gel_conc):
    D_water = D_water_25.get(ion, 1e-9)
    eta_rel = get_viscosity(gel_type, gel_conc)
    T_K = T_C + 273.15
    Ea_R = 18000.0 / 8.314
    factor = np.exp(Ea_R * (1/298.15 - 1/T_K))
    return D_water * factor / eta_rel

def compute_features_for_training(df):
    df = df.copy()
    df['lnX'] = np.nan
    df['lnX2'] = np.nan
    df['prod_z'] = np.nan
    df['inv_r_avg'] = np.nan
    df['gel_желатин'] = 0
    df['gel_агароза'] = 0
    df['gel_силикагель'] = 0
    df['I_agar_lnX'] = 0.0

    for idx, row in df.iterrows():
        gel_raw = row['Гель']
        gel_type, gel_conc = parse_gel(gel_raw)
        T_C = row['T_C'] if pd.notna(row['T_C']) else 22.0
        system = row['Система']
        if system not in mapping:
            continue
        cation, anion = mapping[system]
        if pd.isna(cation) or pd.isna(anion):
            continue
        C_in = row['C_in_M']
        C_out = row['C_out_M']
        if pd.isna(C_in) or pd.isna(C_out):
            continue
        D_in = compute_D_gel(cation, T_C, gel_type, gel_conc)
        D_out = compute_D_gel(anion, T_C, gel_type, gel_conc)
        nu_in, nu_out = stoichiometry.get(system, (1, 1))
        Xcorr = (D_out * C_out / nu_out) / (D_in * C_in / nu_in) if (D_in * C_in) > 0 else np.nan
        if Xcorr > 0:
            lnX = np.log(Xcorr)
            df.at[idx, 'lnX'] = lnX
            df.at[idx, 'lnX2'] = lnX**2
        z_cat = ion_charges.get(cation, 1)
        z_an = ion_charges.get(anion, 1)
        df.at[idx, 'prod_z'] = abs(z_cat * z_an)
        r_cat = ion_radii.get(cation, 1.0)
        r_an = ion_radii.get(anion, 1.5)
        r_avg = (r_cat + r_an) / 2.0
        df.at[idx, 'inv_r_avg'] = 1.0 / r_avg
        if gel_type == 'желатин':
            df.at[idx, 'gel_желатин'] = 1
        elif gel_type == 'агароза':
            df.at[idx, 'gel_агароза'] = 1
        elif gel_type == 'силикагель':
            df.at[idx, 'gel_силикагель'] = 1
        if gel_type == 'агароза' and 'lnX' in df.columns and not pd.isna(df.at[idx, 'lnX']):
            df.at[idx, 'I_agar_lnX'] = df.at[idx, 'lnX']
    df['pH'] = pd.to_numeric(df['pH'], errors='coerce').fillna(7.0)
    df['E_В_м'] = pd.to_numeric(df['E_В_м'], errors='coerce').fillna(0.0)
    df['T_C'] = pd.to_numeric(df['T_C'], errors='coerce').fillna(22.0)
    df['Кольца'] = pd.to_numeric(df['Кольца'], errors='coerce')
    df['p'] = pd.to_numeric(df['p'], errors='coerce')
    return df

df_feat = compute_features_for_training(df)
print("✅ Признаки для обучения рассчитаны.")
df_clean = df_feat.dropna(subset=['lnX', 'lnX2', 'prod_z', 'inv_r_avg', 'pH', 'E_В_м', 'T_C', 'Кольца'])
print(f"После очистки осталось {df_clean.shape[0]} экспериментов.")
n_pos = (df_clean['Кольца'] == 1).sum()
n_neg = (df_clean['Кольца'] == 0).sum()
print(f"Классы: положительных = {n_pos}, отрицательных = {n_neg}")

# =========================== 4. ПОДГОТОВКА ДАННЫХ ===========================
feature_cols = ['lnX', 'lnX2', 'prod_z', 'inv_r_avg', 'pH', 'E_В_м', 'T_C',
                'gel_желатин', 'gel_агароза', 'gel_силикагель', 'I_agar_lnX']
df_reg = df_clean[(df_clean['Кольца']==1) & (df_clean['p'].notna())].copy()
print(f"Регрессия: {len(df_reg)} точек с известным p.")

# =========================== 5. ВАЛИДАЦИЯ ===========================
systems = df_clean['Система'].unique()
print(f"Всего уникальных систем: {len(systems)}")

# Классификация с балансировкой
y_true_class, y_pred_class, y_proba_class = [], [], []
for system in systems:
    test_idx = df_clean[df_clean['Система'] == system].index
    train_idx = df_clean[~df_clean['Система'].isin([system])].index
    if len(train_idx) == 0:
        continue
    X_train = df_clean.loc[train_idx, feature_cols]
    y_train = df_clean.loc[train_idx, 'Кольца'].astype(int)
    X_test = df_clean.loc[test_idx, feature_cols]
    y_test = df_clean.loc[test_idx, 'Кольца'].astype(int)
    clf = LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', solver='lbfgs', max_iter=1000)
    clf.fit(X_train, y_train)
    y_pred_class.extend(clf.predict(X_test))
    y_proba_class.extend(clf.predict_proba(X_test)[:,1])
    y_true_class.extend(y_test)

acc = accuracy_score(y_true_class, y_pred_class)
prec = precision_score(y_true_class, y_pred_class, zero_division=0)
rec = recall_score(y_true_class, y_pred_class)
auc = roc_auc_score(y_true_class, y_proba_class)
cm = confusion_matrix(y_true_class, y_pred_class)
print("\n=== ОБЩАЯ МОДЕЛЬ (все гели) ===")
print(f"Классификация: Accuracy={acc:.3f}, Precision={prec:.3f}, Recall={rec:.3f}, AUC={auc:.3f}")
print("Матрица ошибок (истина\\предсказание):")
print(cm)

# Регрессия
reg_features = ['lnX', 'lnX2', 'E_В_м']
y_true_reg, y_pred_reg = [], []
systems_reg = df_reg['Система'].unique()
for system in systems_reg:
    test_idx = df_reg[df_reg['Система'] == system].index
    train_idx = df_reg[~df_reg['Система'].isin([system])].index
    if len(test_idx)==0 or len(train_idx)<3:
        continue
    Xr_train = df_reg.loc[train_idx, reg_features]
    yr_train = df_reg.loc[train_idx, 'p']
    Xr_test = df_reg.loc[test_idx, reg_features]
    yr_test = df_reg.loc[test_idx, 'p']
    reg = LinearRegression()
    reg.fit(Xr_train, yr_train)
    y_true_reg.extend(yr_test.values)
    y_pred_reg.extend(reg.predict(Xr_test))
mae = mean_absolute_error(y_true_reg, y_pred_reg)
r2 = r2_score(y_true_reg, y_pred_reg)
mape = np.mean(np.abs((np.array(y_true_reg) - np.array(y_pred_reg)) / np.array(y_true_reg))) * 100
print(f"Регрессия: MAE={mae:.4f}, MAPE={mape:.2f}%, R²={r2:.3f}")

# Финальные модели
clf_final = LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', solver='lbfgs', max_iter=1000)
clf_final.fit(df_clean[feature_cols], df_clean['Кольца'].astype(int))
reg_final = LinearRegression()
reg_final.fit(df_reg[reg_features], df_reg['p'])

# =========================== 6. ВЫВОД КОЭФФИЦИЕНТОВ ===========================
print("\n=== КОЭФФИЦИЕНТЫ ОБУЧЕННОЙ МОДЕЛИ ===")
print("\nКлассификатор (логистическая регрессия):")
print(f"{'Признак':<20} {'Коэффициент':>12}")
for i, name in enumerate(feature_cols):
    print(f"{name:<20} {clf_final.coef_[0][i]:12.4f}")
print(f"{'Intercept':<20} {clf_final.intercept_[0]:12.4f}")

print("\nРегрессия (spacing coefficient p):")
print(f"{'Коэффициент':<10} {'Значение':>12}")
print(f"{'Intercept':<10} {reg_final.intercept_:12.4f}")
for i, name in enumerate(reg_features):
    print(f"{name:<10} {reg_final.coef_[i]:12.4f}")

# =========================== 7. ФУНКЦИЯ ПРЕДСКАЗАНИЯ (обученная модель) ===========================
def predict_from_user_input(D_in, D_out, r_in, r_out, z_in, z_out,
                            C_in, C_out, gel_type, gel_conc, T_C, pH, E,
                            nu_in, nu_out):
    eta_rel = get_viscosity(gel_type, gel_conc)
    T_K = T_C + 273.15
    Ea_R = 18000.0 / 8.314
    factor = np.exp(Ea_R * (1/298.15 - 1/T_K))
    D_in_gel = D_in * factor / eta_rel
    D_out_gel = D_out * factor / eta_rel
    Xcorr = (D_out_gel * C_out / nu_out) / (D_in_gel * C_in / nu_in) if (D_in_gel * C_in) > 0 else np.nan
    lnX = np.log(Xcorr)
    lnX2 = lnX**2
    prod_z = z_in * z_out
    r_avg = (r_in + r_out) / 2.0
    inv_r = 1.0 / r_avg
    gel_жел = 1 if gel_type == 'желатин' else 0
    gel_агар = 1 if gel_type == 'агароза' else 0
    gel_силик = 1 if gel_type == 'силикагель' else 0
    I_agar_lnX = lnX if gel_type == 'агароза' else 0.0

    X_new = pd.DataFrame([[lnX, lnX2, prod_z, inv_r, pH, E, T_C,
                           gel_жел, gel_агар, gel_силик, I_agar_lnX]],
                         columns=feature_cols)
    prob = clf_final.predict_proba(X_new)[0, 1]
    if prob >= 0.5:
        X_reg_new = pd.DataFrame([[lnX, lnX2, E]], columns=reg_features)
        p_pred = reg_final.predict(X_reg_new)[0]
        return prob, p_pred
    else:
        return prob, None

def manual_predict(clf_coefs, clf_intercept, reg_coefs, reg_intercept,
                   D_in, D_out, r_in, r_out, z_in, z_out,
                   C_in, C_out, gel_type, gel_conc, T_C, pH, E,
                   nu_in, nu_out):
    eta_rel = get_viscosity(gel_type, gel_conc)
    T_K = T_C + 273.15
    Ea_R = 18000.0 / 8.314
    factor = np.exp(Ea_R * (1/298.15 - 1/T_K))
    D_in_gel = D_in * factor / eta_rel
    D_out_gel = D_out * factor / eta_rel
    Xcorr = (D_out_gel * C_out / nu_out) / (D_in_gel * C_in / nu_in) if (D_in_gel * C_in) > 0 else np.nan
    lnX = np.log(Xcorr)
    lnX2 = lnX**2
    prod_z = z_in * z_out
    r_avg = (r_in + r_out) / 2.0
    inv_r = 1.0 / r_avg
    gel_жел = 1 if gel_type == 'желатин' else 0
    gel_агар = 1 if gel_type == 'агароза' else 0
    gel_силик = 1 if gel_type == 'силикагель' else 0
    I_agar_lnX = lnX if gel_type == 'агароза' else 0.0

    z = (clf_intercept +
         clf_coefs[0] * lnX +
         clf_coefs[1] * lnX2 +
         clf_coefs[2] * prod_z +
         clf_coefs[3] * inv_r +
         clf_coefs[4] * pH +
         clf_coefs[5] * E +
         clf_coefs[6] * T_C +
         clf_coefs[7] * gel_жел +
         clf_coefs[8] * gel_агар +
         clf_coefs[9] * gel_силик +
         clf_coefs[10] * I_agar_lnX)
    prob = 1 / (1 + np.exp(-z))
    if prob >= 0.5:
        p_pred = (reg_intercept +
                  reg_coefs[0] * lnX +
                  reg_coefs[1] * lnX2 +
                  reg_coefs[2] * E)
        return prob, p_pred
    else:
        return prob, None

# =========================== 8. ГРАФИЧЕСКИЙ ИНТЕРФЕЙС ===========================
display(HTML("<style>.widget-label { color: black; } .widget-text input, .widget-float input { background-color: white; color: black; }</style>"))

print("\n" + "="*70)
print("ИНТЕРАКТИВНОЕ ПРЕДСКАЗАНИЕ")
print("="*70)

mode = widgets.RadioButtons(
    options=['Использовать обученную модель', 'Ручной расчёт (свои коэффициенты)'],
    value='Использовать обученную модель',
    description='Режим:',
    style={'description_width': 'initial'}
)

clf_coefs_manual = []
for i in range(11):
    clf_coefs_manual.append(widgets.FloatText(value=0.0, description=f'β_{i}', layout=widgets.Layout(width='80px')))
clf_intercept_manual = widgets.FloatText(value=0.0, description='Intercept (класс)')
reg_coefs_manual = []
for i in range(3):
    reg_coefs_manual.append(widgets.FloatText(value=0.0, description=f'β_{i} рег', layout=widgets.Layout(width='80px')))
reg_intercept_manual = widgets.FloatText(value=0.0, description='Intercept (рег)')

manual_coeffs_box = widgets.VBox([
    widgets.HTML("<b>Коэффициенты классификатора (11 признаков в порядке feature_cols):</b>"),
    widgets.HBox(clf_coefs_manual),
    clf_intercept_manual,
    widgets.HTML("<b>Коэффициенты регрессии (lnX, lnX2, E):</b>"),
    widgets.HBox(reg_coefs_manual),
    reg_intercept_manual
])

def create_inputs():
    D_in = widgets.FloatText(value=1.65e-9, description='D_water иона ВНУТРИ (м²/с):', step=1e-10, style={'description_width': 'initial'})
    D_out = widgets.FloatText(value=1.00e-9, description='D_water иона СНАРУЖИ (м²/с):', step=1e-10, style={'description_width': 'initial'})
    r_in = widgets.FloatText(value=1.15, description='Радиус иона ВНУТРИ (Å):', step=0.01, style={'description_width': 'initial'})
    r_out = widgets.FloatText(value=2.50, description='Радиус иона СНАРУЖИ (Å):', step=0.01, style={'description_width': 'initial'})
    z_in = widgets.IntText(value=1, description='Заряд иона ВНУТРИ:', style={'description_width': 'initial'})
    z_out = widgets.IntText(value=2, description='Заряд иона СНАРУЖИ:', style={'description_width': 'initial'})
    C_in = widgets.FloatText(value=0.1, description='C_in (М):')
    C_out = widgets.FloatText(value=0.1, description='C_out (М):')
    gel_type = widgets.Dropdown(options=['желатин', 'агароза', 'силикагель'], value='желатин', description='Тип геля:')
    gel_conc = widgets.FloatText(value=5.0, description='Конц. геля (%):')
    T = widgets.FloatText(value=22.0, description='Температура (°C):')
    pH = widgets.FloatText(value=7.0, description='pH:')
    E = widgets.FloatText(value=0.0, description='E (В/м):')
    nu_in = widgets.FloatText(value=1.0, description='nu_in (стех. коэф. иона ВНУТРИ):', step=0.1)
    nu_out = widgets.FloatText(value=1.0, description='nu_out (стех. коэф. иона СНАРУЖИ):', step=0.1)
    return (D_in, D_out, r_in, r_out, z_in, z_out,
            C_in, C_out, gel_type, gel_conc, T, pH, E,
            nu_in, nu_out)

(D_in, D_out, r_in, r_out, z_in, z_out,
 C_in, C_out, gel_type, gel_conc, T, pH, E,
 nu_in, nu_out) = create_inputs()

button = widgets.Button(description='ПРЕДСКАЗАТЬ', button_style='success')
output = widgets.Output()

def on_predict(b):
    with output:
        clear_output()
        try:
            if mode.value == 'Использовать обученную модель':
                prob, p_pred = predict_from_user_input(
                    D_in.value, D_out.value,
                    r_in.value, r_out.value,
                    z_in.value, z_out.value,
                    C_in.value, C_out.value,
                    gel_type.value, gel_conc.value,
                    T.value, pH.value, E.value,
                    nu_in.value, nu_out.value
                )
            else:
                clf_coefs = [w.value for w in clf_coefs_manual]
                clf_intercept = clf_intercept_manual.value
                reg_coefs = [w.value for w in reg_coefs_manual]
                reg_intercept = reg_intercept_manual.value
                prob, p_pred = manual_predict(
                    clf_coefs, clf_intercept, reg_coefs, reg_intercept,
                    D_in.value, D_out.value,
                    r_in.value, r_out.value,
                    z_in.value, z_out.value,
                    C_in.value, C_out.value,
                    gel_type.value, gel_conc.value,
                    T.value, pH.value, E.value,
                    nu_in.value, nu_out.value
                )
            if prob >= 0.5:
                print(f"\n✅ Вероятность образования колец: {prob:.3f}")
                if p_pred is not None:
                    print(f"✅ Spacing coefficient p = {p_pred:.3f}")
                else:
                    print("⚠️ Регрессия не применялась (вероятность < 0.5)")
            else:
                print(f"\n❌ Вероятность образования колец: {prob:.3f}")
                print("❌ Кольца НЕ образуются (вероятность ниже 0.5)")
        except Exception as e:
            print(f"Ошибка: {e}. Проверьте правильность ввода чисел.")

button.on_click(on_predict)

def update_visibility(change):
    if mode.value == 'Ручной расчёт (свои коэффициенты)':
        manual_coeffs_box.layout.visibility = 'visible'
    else:
        manual_coeffs_box.layout.visibility = 'hidden'

mode.observe(update_visibility, 'value')
update_visibility(None)

left = widgets.VBox([D_in, D_out, r_in, r_out, z_in, z_out, nu_in, nu_out])
right = widgets.VBox([C_in, C_out, gel_type, gel_conc, T, pH, E])
common_inputs = widgets.HBox([left, right])
all_widgets = widgets.VBox([mode, common_inputs, manual_coeffs_box, button, output])

display(all_widgets)
print("\n✅ Модель обучена. Коэффициенты выведены выше. Введите все параметры и нажмите кнопку.")

Загрузите ваш CSV-файл (liesegang_dataset.csv или .xlsx):


Saving liesegang_dataset.xlsx to liesegang_dataset.xlsx
✅ Загружено 213 строк.
✅ Признаки для обучения рассчитаны.
После очистки осталось 210 экспериментов.
Классы: положительных = 197, отрицательных = 13
Регрессия: 132 точек с известным p.
Всего уникальных систем: 72

=== ОБЩАЯ МОДЕЛЬ (все гели) ===
Классификация: Accuracy=0.776, Precision=0.963, Recall=0.792, AUC=0.636
Матрица ошибок (истина\предсказание):
[[  7   6]
 [ 41 156]]
Регрессия: MAE=0.0213, MAPE=2.00%, R²=0.425

=== КОЭФФИЦИЕНТЫ ОБУЧЕННОЙ МОДЕЛИ ===

Классификатор (логистическая регрессия):
Признак               Коэффициент
lnX                       -1.3530
lnX2                       0.7809
prod_z                    -0.1920
inv_r_avg                 -0.1303
pH                         0.3822
E_В_м                      0.0570
T_C                        0.0137
gel_желатин                0.7290
gel_агароза               -1.6387
gel_силикагель            -0.4951
I_agar_lnX                 0.7055
Intercept                 -1.985


ИНТЕРАКТИВНОЕ ПРЕДСКАЗАНИЕ



✅ Модель обучена. Коэффициенты выведены выше. Введите все параметры и нажмите кнопку.
